# DSA210 — Machine learning: race action

This notebook loads `data/processed/race_level_data.csv` (build it by running `01_data_eda_hypothesis.ipynb` first) and trains leak free models:

1. **Classification** predict whether a race is high action (top quartile of `mean_abs_pos_change`) from circuit, weather, field size, and season metadata.
2. **Regression** predict continuous `mean_abs_pos_change` from the same predictors.

Predictors **exclude** grid-to-finish summaries (`mean_abs_pos_change`, `total_abs_pos_change`, etc.) so we are not labeling with the same information we pretend to predict.

Optional in race add-ons (`n_pit_stops`, `n_compounds_used`) can be toggled on (they improve fit partly because they are concurrent with on-track action).

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJ / "src"
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

RACE_CSV = PROJ / "data" / "processed" / "race_level_data.csv"

if not RACE_CSV.exists():
    raise FileNotFoundError(
        f"Missing {RACE_CSV}. Run notebooks/01_data_eda_hypothesis.ipynb to build the dataset."
    )

race_df = pd.read_csv(RACE_CSV)
print(f"Loaded {len(race_df)} races, {race_df.shape[1]} columns")

Loaded 148 races, 26 columns


In [2]:
from features import label_high_action_races

# Match EDA notebook: top 25 % of mean absolute grid finish change
race_df = label_high_action_races(race_df, action_col="mean_abs_pos_change", quantile=0.75)
thr = race_df.attrs.get("high_action_threshold", float("nan"))
print(f"High-action threshold (≥{thr:.2f}): {race_df['high_action'].sum()} / {len(race_df)} races")

High-action threshold (≥4.30): 40 / 148 races


In [3]:
from models import (
    assert_no_outcome_leak,
    cross_val_r2_regressor,
    cross_val_roc_auc_classifier,
    fit_rf_classifier,
    fit_rf_regressor,
    make_train_test,
    modeling_columns,
    modeling_matrix,
    rf_feature_importance_df,
)

INCLUDE_PITS = False  # set True to add pit / compound counts (stronger signal partly inrace)

feat_cols = modeling_columns(include_in_race_features=INCLUDE_PITS)
assert_no_outcome_leak(feat_cols)
print("Features (%d):" % len(feat_cols), ", ".join(feat_cols))

Features (17): year, round, n_starters, n_classified, n_dnf, n_laps, avg_air_temp, avg_track_temp, avg_humidity, avg_wind_speed, rain_pct, any_rain, circuit_length_km, circuit_turns, circuit_type, location, country


In [4]:
# --- Classification: high_action ---
Xc, yc = modeling_matrix(race_df, "high_action", include_in_race_features=INCLUDE_PITS)
cv_auc = cross_val_roc_auc_classifier(Xc, yc, n_splits=5)
print(f"Stratified 5-fold CV mean ROC-AUC (leak-free): {cv_auc:.3f}")

split_c = make_train_test(Xc, yc, stratify=True)
clf_pipe, clf_metrics = fit_rf_classifier(split_c)
print("Hold-out ROC-AUC:", round(clf_metrics["roc_auc"], 3))
print("Hold-out precision/recall (class 1):", clf_metrics["report"]["1"])

Stratified 5-fold CV mean ROC-AUC (leak-free): 0.705


Hold-out ROC-AUC: 0.722
Hold-out precision/recall (class 1): {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 8.0}


In [5]:
# --- Regression: mean_abs_pos_change ---
Xr, yr = modeling_matrix(race_df, "mean_abs_pos_change", include_in_race_features=INCLUDE_PITS)
cv_r2 = cross_val_r2_regressor(Xr, yr, n_splits=5)
print(f"5-fold CV mean R² (leak-free): {cv_r2:.3f}")

split_r = make_train_test(Xr, yr, stratify=False)
reg_pipe, reg_metrics = fit_rf_regressor(split_r)
print("Hold-out R²:", round(reg_metrics["r2"], 3))
print("Hold-out MAE (positions):", round(reg_metrics["mae"], 3))

5-fold CV mean R² (leak-free): 0.144
Hold-out R²: 0.237
Hold-out MAE (positions): 0.917


In [6]:
# Random Forest importances (one-hot expanded) when features correlate
imp_c = rf_feature_importance_df(clf_pipe, top=15)
imp_r = rf_feature_importance_df(reg_pipe, top=15)
print("Classifier — top features\n", imp_c.to_string(index=False))
print("\nRegressor — top features\n", imp_r.to_string(index=False))

Classifier — top features
                 feature  importance
             num__n_dnf    0.197220
    num__avg_track_temp    0.094240
      num__avg_air_temp    0.090373
      num__avg_humidity    0.062641
              num__year    0.059474
 num__circuit_length_km    0.059044
     num__circuit_turns    0.058075
    num__avg_wind_speed    0.054207
             num__round    0.049568
            num__n_laps    0.042389
          num__rain_pct    0.027996
cat__country_Azerbaijan    0.019877
     cat__location_Baku    0.017062
     cat__country_Italy    0.016734
      num__n_classified    0.013734

Regressor — top features
                 feature  importance
             num__n_dnf    0.331111
    num__avg_track_temp    0.112723
      num__avg_air_temp    0.110573
      num__avg_humidity    0.078957
    num__avg_wind_speed    0.071919
             num__round    0.059918
              num__year    0.047320
     num__circuit_turns    0.038229
            num__n_laps    0.037123
 num__circ